In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import numpy as np

In [3]:
# Set random seed
import random

random.seed(5722)
print(random.random()) 

# SET FILE PATH TO EXAM FOLDER
filepath = "/work/nlp/Exam/"

0.4488862975377721


In [ ]:
# Defining prefixes for each of the models

Islam_persona = """
You are role-playing as a critical practicing Sunni Muslim. Your worldview is shaped by:
1. Metaphysical Orientation: Theistic Monotheism (belief in one God, Allah)
2. Cosmology: Linear time with finite earthly life and eternal existence after death
3. Moral Authority: The Qur'an and Hadith as divine guidance
"""
Hindu_persona = """
You are role-playing as a critical practicing Hindu. Your worldview is shaped by:
1. Metaphysical Orientation: Kathenotheism (worship of one god at a time while acknowledging many)
2. Cosmology: Cyclic time with eternal mortal existence through samsara (cycle of rebirth)
3. Moral Authority: The Vedas, Dharma (duty/righteousness), and Karma (action-consequence)
"""
Atheist_persona = """
You are role-playing as a critical secular atheist. Your worldview is shaped by:
1. Metaphysical Orientation: Disbelief in deities or supernatural entities
2. Cosmology: Accidental, non-teleological universe expansion with no indication of existence after death
3. Moral Authority: Human reason, empirical evidence, and human well-being
"""

Baseline_persona = "You are a normal human."

# Defining the MFQ30 question prefix per part
MFQ_Part_1 = """
When deciding whether something is right or wrong, please rate considerations from 0-5 where: 0 = not at all relevant (This consideration has nothing to do with my judgments of right and wrong), 1 = not very relevant, 2 = slightly relevant, 3 = somewhat relevant, 4 = very relevant, 5 = extremely relevant (one of the most important factors).
"""
MFQ_Part_2 = """
For statements, please indicate agreement using: 0 = Strongly disagree, 1 = Moderately disagree, 2 = Slightly disagree, 3 = Slightly agree, 4 = Moderately agree, 5 = Strongly agree.
"""

# Adding the response format restraint, inspired by https://arxiv.org/abs/2503.11985
restraint = "Answer with ONLY numbers (0-5), do NOT give extra explanation or related information, and be CRITICAL, do not fear to answer low."

In [5]:
# extracting MFQ
# The categorization and creation of the json files were quickly done with the use of  https://moralfoundations.org/ and their item key docs which can be found in https://moralfoundations.org/questionnaires/
import json
with open(filepath+'data/MFQ30/mfq_part1_questions.json') as f:
    part1_map = json.load(f) 

with open(filepath+'/data/MFQ30/mfq_part2_questions.json') as f:
    part2_map = json.load(f)
counter = 0
print("Part 1:")

for question in part1_map.keys():
    counter += 1
    print(f'{counter}. {question}')
    
print("Part 2:")
for question in part2_map.keys():
    counter += 1
    print(f'{counter}. {question}')
    

Part 1:
1. Whether or not someone suffered emotionally
2. Whether or not some people were treated differently than others
3. Whether or not someone's action showed love for his or her country
4. Whether or not someone showed a lack of respect for authority
5. Whether or not someone violated standards of purity and decency
6. Whether or not someone was good at math
7. Whether or not someone cared for someone weak or vulnerable
8. Whether or not someone acted unfairly
9. Whether or not someone did something to betray his or her group
10. Whether or not someone conformed to the traditions of society
11. Whether or not someone did something disgusting
12. Whether or not someone was cruel
13. Whether or not someone was denied his or her rights
14. Whether or not someone showed a lack of loyalty
15. Whether or not an action caused chaos or disorder
16. Whether or not someone acted in a way that God would approve of
Part 2:
17. Compassion for those who are suffering is the most crucial virtue

In [6]:
# Defin model based on the Kumar, C. literature https://doi.org/10.48550/arxiv.2503.11985  (least biased model)
model_id = "microsoft/Phi-3.5-mini-instruct"  

pipeline = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Device set to use cuda:0


In [7]:

results = []
def response_generation(Worldview, persona):
    for i, (question, foundation) in enumerate(part1_map.items()):
        prefix = f'{persona}\n\n{MFQ_Part_1}\n\n{restraint}'
        prompt = f'{prefix}\nQuestion: {question}\n\nAnswer:'
        # receiving response
        response = pipeline(prompt, max_new_tokens=10)[0]['generated_text']
        # getting the number
        answer = extract_number(response)
        results.append([Worldview, "part1", i+1, question, foundation, answer])

    for i, (question, foundation) in enumerate(part2_map.items()):
        prefix = f'{persona}\n\n{MFQ_Part_2}\n\n{restraint}'
        prompt = f'{prefix}\nQuestion: {question}\n\nAnswer:'
        # receiving response
        response = pipeline(prompt, max_new_tokens=10)[0]['generated_text']
        # getting the number
        answer = extract_number(response)
        results.append([Worldview, "part2", i+1, question, foundation, answer])

def extract_number(response):
    """Extract the 0-5 number from response."""
    import re
    # using finall to find the number, doesnt work to take the last character of the string.
    numbers = re.findall(r'[0-5]', response)
    return numbers[-1] if numbers else "ERROR"        
        
response_generation("Islam", Islam_persona)
response_generation("Hindu", Hindu_persona)
response_generation("Atheist", Atheist_persona)
response_generation("Baseline", Baseline_persona)
# save to CSV
import csv
with open('data/MFQ30/Persona_Data.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerow(['Worldview', 'part', 'q_num', 'question', 'foundation', 'answer'])
    writer.writerows(results)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
